# 04 - Classification and evaluation

Predicting segments from early purchase behaviour without leakage (Phases 6 and 7).

This notebook reads the outputs in `results/` (produced by the scripts in `src/`); it does not repeat the cleaning logic.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
RES = ROOT / "results"
pd.set_option("display.width", 200)

def csv(name):
    path = RES / name
    if not path.exists():
        print(f"{name} not found: run `python src/run_all.py` from the repository root first.")
        return None
    return pd.read_csv(path)

def js(name):
    path = RES / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


## Leakage audit

In [ ]:
audit = js('classification_leakage_audit.json'); print(json.dumps(audit, indent=2))

## Test metrics (the `future` task is the leakage-free headline; `lifetime` is optimistic)

In [ ]:
m = csv('metrics.csv')
if m is not None:
    t = m[m.split == 'test']
    display(t[t.metric.isin(['accuracy','macro_f1','weighted_f1','balanced_accuracy'])].pivot_table(index=['task','model'], columns='metric', values='value').round(3))
    display(t[t.metric.str.startswith('recall[')][['task','model','metric','value']].round(3))
    display(m[(m.split == 'cv_train') & (m.metric == 'macro_f1_mean')][['task','model','value','notes']])

## Confusion matrices (best model by cross-validated macro-F1)

In [ ]:
for task in ['segment_future', 'segment_lifetime']:
    cm = csv(f'confusion_matrix_{task}.csv')
    if cm is not None:
        cm = cm.set_index(cm.columns[0]); print(task); display(cm)

## Random Forest feature importance

In [ ]:
for task in ['segment_future', 'segment_lifetime']:
    fi = csv(f'feature_importance_{task}.csv')
    if fi is not None:
        print(task); display(fi.head(8))